# Vector Search: Exact Search and the Approximate Indexes

The runnable code of [Vector Search: Exact Search and the Approximate Indexes](https://residual-stream.lvtan71.workers.dev/posts/vector-search-and-ann-indexes/), in the order the post runs it. Each
code cell is followed by the output the post printed, so you can compare.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lvtan71/residual-stream-code/blob/main/notebooks/vector-search-and-ann-indexes.ipynb)

**Hardware:** Runs on a CPU; Colab's free runtime is enough. About 6 min end to end.


The post's numbers come from an Apple M4 laptop. Other hardware and library
versions can change the last digits of timings and of some floating-point
results; the conclusions should not change.

In [ ]:
%pip install -q faiss-cpu

# Setup: fetch the helpers and data this post's code uses into a working folder.
import os, shutil, subprocess, sys
SLUG = "vector-search-and-ann-indexes"
ROOT = os.path.abspath("..") if os.path.isdir("../shared") else os.path.abspath("residual-stream-code")
if not os.path.isdir(ROOT):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/lvtan71/residual-stream-code", ROOT], check=True)
WORK = os.path.join(ROOT, "work", SLUG)
os.makedirs(WORK, exist_ok=True)
for f in []:
    shutil.copy(os.path.join(ROOT, f), WORK)
    if f.endswith(".tar.gz"):                       # the blog's posts, as of that run
        shutil.unpack_archive(os.path.join(WORK, os.path.basename(f)), WORK)
os.chdir(WORK)
sys.path.insert(0, WORK)
print("working in", WORK)

## Exact search, and why trees don't save it

[Read this part in the post](https://residual-stream.lvtan71.workers.dev/posts/vector-search-and-ann-indexes/#exact-search-and-why-trees-dont-save-it)

**Not shown in the post:** the post's code searches WikiText-2 sentences embedded beforehand. This cell embeds them (several minutes on a CPU).

In [ ]:
import numpy as np
import pyarrow.parquet as pq
from huggingface_hub import snapshot_download
from sentence_transformers import SentenceTransformer

# The vectors: every sentence of WikiText-2's training set (over 40
# characters), embedded with post 8's all-MiniLM-L6-v2 (384 dimensions,
# normalized), on the CPU. Queries: 1,000 sentences of WikiText-2's test
# set, which none of the vectors contain.
import re
W = snapshot_download("Salesforce/wikitext", repo_type="dataset",
                      revision="b08601e04326c79dfdd32d625aee71d232d685c3") + "/wikitext-2-raw-v1/"
def paragraphs(split):
    lines = pq.read_table(W + f"{split}-00000-of-00001.parquet")["text"].to_pylist()
    paras = [l.strip() for l in lines if l.strip() and not l.strip().startswith("=")]
    return [x for p in paras for x in re.split(r"(?<=[.!?]) (?=[A-Z])", p) if len(x) > 40]
docs, queries = paragraphs("train"), paragraphs("test")[:1000]
enc = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device="cpu")
X = enc.encode(docs, batch_size=128, normalize_embeddings=True, show_progress_bar=False).astype("float32")
Q = enc.encode(queries, batch_size=128, normalize_embeddings=True, show_progress_bar=False).astype("float32")
np.save("docs.npy", X); np.save("queries.npy", Q)
print(f"{len(docs):,} sentences embedded, {X.shape[1]} dimensions; {len(queries):,} queries")

In [ ]:
import numpy as np

# Why trees fail in high dimensions: for 10,000 random points, the nearest
# and the farthest neighbor of a random query end up almost equally far away.
rng = np.random.default_rng(0)
for dim in (2, 10, 100, 384):
    P = rng.normal(size=(10_000, dim)); q = rng.normal(size=dim)
    dist = np.linalg.norm(P - q, axis=1)
    print(f"random points, {dim:3d} dimensions: nearest / farthest distance = {dist.min() / dist.max():.2f}")

# The real sentence embeddings of the next section (384 dimensions), for
# 200 of the query sentences: median of the same ratio.
X, Q = np.load("docs.npy"), np.load("queries.npy")
r = [np.sqrt(np.maximum(2 - 2 * (X @ q).max(), 0)) / np.sqrt(2 - 2 * (X @ q).min()) for q in Q[:200]]
print(f"real embeddings,   384 dimensions: nearest / farthest distance = {np.median(r):.2f} (median of 200 queries)")

The post printed:

```text
random points,   2 dimensions: nearest / farthest distance = 0.00
random points,  10 dimensions: nearest / farthest distance = 0.19
random points, 100 dimensions: nearest / farthest distance = 0.61
random points, 384 dimensions: nearest / farthest distance = 0.78
real embeddings,   384 dimensions: nearest / farthest distance = 0.63 (median of 200 queries)
```

## Measured: recall, speed and memory

[Read this part in the post](https://residual-stream.lvtan71.workers.dev/posts/vector-search-and-ann-indexes/#measured-recall-speed-and-memory)

In [ ]:
import time
import faiss
import numpy as np

# Exact and approximate nearest-neighbor search over 71,865 real sentence
# embeddings (384 dimensions, normalized), 1,000 held-out queries, top 10.
# Truth: exact inner-product search. Every index is queried one query at a
# time on one CPU thread, the latency a single request sees; each setting
# is timed three times and the fastest kept, since other work on the
# machine only ever slows a run down.
faiss.omp_set_num_threads(1)
X, Q = np.load("docs.npy"), np.load("queries.npy")
N, d, k = *X.shape, 10
exact = faiss.IndexFlatIP(d); exact.add(X)
_, truth = exact.search(Q, k)

# On unit vectors, L2, dot product and cosine rank identically:
#    |q - x|^2 = 2 - 2 q.x.
l2 = faiss.IndexFlatL2(d); l2.add(X); _, by_l2 = l2.search(Q[:100], k)
same = all(set(a) == set(b) for a, b in zip(by_l2, truth[:100]))
swaps = [(i, j) for i in range(100) for j in range(k) if by_l2[i, j] != truth[i, j]]
ties = all(np.isclose(Q[i] @ X[by_l2[i, j]], Q[i] @ X[truth[i, j]]) for i, j in swaps)
print(f"L2 vs dot product, 100 queries: same top-10 sets: {same}; order differs at {len(swaps)} positions, "
      f"all between exactly tied scores (duplicate sentences): {ties}")

def measure(index, name, bytes_per_vector, setter=None, values=(None,)):
    for v in values:
        if setter:
            setter(v)
        times = []
        for _ in range(3):
            t0 = time.perf_counter()
            found = np.vstack([index.search(Q[i:i + 1], k)[1] for i in range(len(Q))])
            times.append(time.perf_counter() - t0)
        ms = min(times) / len(Q) * 1e3
        recall = np.mean([len(set(f) & set(t)) / k for f, t in zip(found, truth)])
        label = name if v is None else f"{name}, {v}"
        print(f"{label:32s} recall@10 {recall:6.3f}   {ms:6.3f} ms/query   {bytes_per_vector:5.0f} bytes/vector")

print(f"\n{N:,} vectors, {d} dimensions:")
measure(exact, "flat (exact)", 4 * d)
t0 = time.perf_counter()
ivf = faiss.IndexIVFFlat(faiss.IndexFlatIP(d), d, 256, faiss.METRIC_INNER_PRODUCT); ivf.train(X); ivf.add(X)
print(f"  (IVF, 256 lists: built in {time.perf_counter() - t0:.1f} s)")
measure(ivf, "IVF 256 lists, probe", 4 * d + 8, lambda v: setattr(ivf, "nprobe", v), (1, 4, 16, 64))
# FAISS's guideline is 4 to 16 times sqrt(N) lists; sqrt(71,865) is 268.
ivf4 = faiss.IndexIVFFlat(faiss.IndexFlatIP(d), d, 1024, faiss.METRIC_INNER_PRODUCT); ivf4.train(X); ivf4.add(X)
measure(ivf4, "IVF 1024 lists, probe", 4 * d + 8, lambda v: setattr(ivf4, "nprobe", v), (8, 16, 32, 64))
t0 = time.perf_counter()
pq = faiss.IndexPQ(d, 48, 8, faiss.METRIC_INNER_PRODUCT); pq.train(X); pq.add(X)
print(f"  (PQ, 48 sub-vectors x 8 bits: built in {time.perf_counter() - t0:.1f} s)")
measure(pq, "PQ, 48-byte codes", 48)
ivfpq = faiss.IndexIVFPQ(faiss.IndexFlatIP(d), d, 256, 48, 8, faiss.METRIC_INNER_PRODUCT); ivfpq.train(X); ivfpq.add(X)
measure(ivfpq, "IVF-PQ, probe", 48 + 8, lambda v: setattr(ivfpq, "nprobe", v), (4, 16, 64))
# PQ's recall is capped by its coarse codes; re-scoring the top 100
# candidates with the exact vectors (kept in memory, or on disk) lifts it.
base = faiss.IndexIVFPQ(faiss.IndexFlatIP(d), d, 256, 48, 8, faiss.METRIC_INNER_PRODUCT); base.train(X)
refine = faiss.IndexRefineFlat(base); refine.k_factor = 10; refine.add(X)
measure(refine, "IVF-PQ + re-rank 100, probe", 48 + 8 + 4 * d, lambda v: setattr(base, "nprobe", v), (16, 64))
t0 = time.perf_counter()
hnsw = faiss.IndexHNSWFlat(d, 32, faiss.METRIC_INNER_PRODUCT); hnsw.hnsw.efConstruction = 128; hnsw.add(X)
print(f"  (HNSW, M = 32: built in {time.perf_counter() - t0:.1f} s)")
measure(hnsw, "HNSW M=32, efSearch", 4 * d + 2 * 32 * 4, lambda v: setattr(hnsw.hnsw, "efSearch", v), (16, 32, 64, 128))
lsh = faiss.IndexLSH(d, 1024); lsh.add(X)
measure(lsh, "LSH, 1024-bit codes, scanned", 1024 / 8)

The post printed:

```text
L2 vs dot product, 100 queries: same top-10 sets: True; order differs at 4 positions, all between exactly tied scores (duplicate sentences): True

71,865 vectors, 384 dimensions:
flat (exact)                     recall@10  1.000    4.071 ms/query    1536 bytes/vector
  (IVF, 256 lists: built in 0.5 s)
IVF 256 lists, probe, 1          recall@10  0.496    0.026 ms/query    1544 bytes/vector
IVF 256 lists, probe, 4          recall@10  0.812    0.076 ms/query    1544 bytes/vector
IVF 256 lists, probe, 16         recall@10  0.955    0.279 ms/query    1544 bytes/vector
IVF 256 lists, probe, 64         recall@10  0.994    1.098 ms/query    1544 bytes/vector
IVF 1024 lists, probe, 8         recall@10  0.799    0.062 ms/query    1544 bytes/vector
IVF 1024 lists, probe, 16        recall@10  0.886    0.098 ms/query    1544 bytes/vector
IVF 1024 lists, probe, 32        recall@10  0.941    0.174 ms/query    1544 bytes/vector
IVF 1024 lists, probe, 64        recall@10  0.974    0.293 ms/query    1544 bytes/vector
  (PQ, 48 sub-vectors x 8 bits: built in 13.6 s)
PQ, 48-byte codes                recall@10  0.626    0.835 ms/query      48 bytes/vector
IVF-PQ, probe, 4                 recall@10  0.568    0.067 ms/query      56 bytes/vector
IVF-PQ, probe, 16                recall@10  0.604    0.118 ms/query      56 bytes/vector
IVF-PQ, probe, 64                recall@10  0.608    0.291 ms/query      56 bytes/vector
IVF-PQ + re-rank 100, probe, 16  recall@10  0.942    0.162 ms/query    1592 bytes/vector
IVF-PQ + re-rank 100, probe, 64  recall@10  0.969    0.328 ms/query    1592 bytes/vector
  (HNSW, M = 32: built in 47.4 s)
HNSW M=32, efSearch, 16          recall@10  0.876    0.089 ms/query    1792 bytes/vector
HNSW M=32, efSearch, 32          recall@10  0.950    0.153 ms/query    1792 bytes/vector
HNSW M=32, efSearch, 64          recall@10  0.983    0.279 ms/query    1792 bytes/vector
HNSW M=32, efSearch, 128         recall@10  0.994    0.551 ms/query    1792 bytes/vector
LSH, 1024-bit codes, scanned     recall@10  0.631    0.364 ms/query     128 bytes/vector
```